In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, RidgeClassifier, Lasso, ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, make_scorer
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ============================================================================
# 1. DATA LOADING AND PREPROCESSING
# ============================================================================
def load_and_preprocess_data(train_path, test_path=None):
    """Load and preprocess embeddings data"""
    print("Loading training data...")
    with open(train_path, 'r') as f:
        train_data = json.load(f)
    
    X_train = []
    y_train = []
    for record in train_data:
        features = record['image_embedding'] + record['text_embedding']
        X_train.append(features)
        y_train.append(record['label'])
    
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    print(f"Training samples: {len(X_train)}")
    print(f"Feature dimension: {X_train.shape[1]}")
    print(f"Class distribution: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"Class imbalance ratio: {sum(y_train==0)/sum(y_train==1):.2f}:1")
    
    test_ids = None
    X_test = None
    if test_path:
        print("\nLoading test data...")
        with open(test_path, 'r') as f:
            test_data = json.load(f)
        
        X_test = []
        test_ids = []
        for record in test_data:
            features = record['image_embedding'] + record['text_embedding']
            X_test.append(features)
            test_ids.append(record['id'])
        
        X_test = np.array(X_test)
        print(f"Test samples: {len(X_test)}")
    
    return X_train, y_train, X_test, test_ids

# ============================================================================
# 2. ENHANCED LOGISTIC REGRESSION TUNING
# ============================================================================
def train_logistic_regression_extensive(X_train, y_train, X_val, y_val, search_type='grid'):
    """
    Extensive hyperparameter tuning for Logistic Regression
    search_type: 'grid' or 'random'
    """
    print("\n" + "="*80)
    print("EXTENSIVE LOGISTIC REGRESSION HYPERPARAMETER TUNING")
    print("="*80)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Comprehensive parameter grid
    param_grid = {
        'C': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0],
        'penalty': ['l1', 'l2', 'elasticnet'],
        'solver': ['liblinear', 'saga', 'lbfgs'],
        'max_iter': [500, 1000, 2000, 3000],
        'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}],
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]  # For elasticnet
    }
    
    # Random search for faster exploration
    if search_type == 'random':
        param_distributions = {
            'C': np.logspace(-4, 2, 50),
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['liblinear', 'saga', 'lbfgs'],
            'max_iter': [500, 1000, 2000, 3000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}],
            'l1_ratio': np.linspace(0.1, 0.9, 9)
        }
        
        lr = LogisticRegression(random_state=42)
        search = RandomizedSearchCV(
            lr, param_distributions,
            n_iter=200,  # Number of random combinations to try
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=2,
            random_state=42
        )
    else:
        # Grid search with constraint handling
        print("Note: Using constrained grid search due to solver-penalty compatibility")
        
        # Create compatible parameter combinations
        lr = LogisticRegression(random_state=42)
        
        # Separate grids for different solvers
        param_grid_lbfgs = {
            'C': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
            'penalty': ['l2', 'none'],
            'solver': ['lbfgs'],
            'max_iter': [500, 1000, 2000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}]
        }
        
        param_grid_saga = {
            'C': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['saga'],
            'max_iter': [1000, 2000, 3000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
        }
        
        param_grid_liblinear = {
            'C': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'],
            'max_iter': [500, 1000, 2000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}]
        }
        
        # Run grid search for each solver
        best_score = 0
        best_params = None
        best_estimator = None
        
        for solver_name, pg in [('lbfgs', param_grid_lbfgs), 
                                 ('saga', param_grid_saga), 
                                 ('liblinear', param_grid_liblinear)]:
            print(f"\nSearching with {solver_name} solver...")
            
            search = GridSearchCV(
                lr, pg,
                cv=5,
                scoring='f1_macro',
                n_jobs=-1,
                verbose=1
            )
            
            search.fit(X_train_scaled, y_train)
            
            if search.best_score_ > best_score:
                best_score = search.best_score_
                best_params = search.best_params_
                best_estimator = search.best_estimator_
        
        # Create a mock search object for consistency
        class MockSearch:
            def __init__(self, best_params, best_score, best_estimator, cv_results):
                self.best_params_ = best_params
                self.best_score_ = best_score
                self.best_estimator_ = best_estimator
                self.cv_results_ = cv_results
        
        search = MockSearch(best_params, best_score, best_estimator, search.cv_results_)
    
    print(f"\n{'='*80}")
    print("LOGISTIC REGRESSION - BEST PARAMETERS")
    print(f"{'='*80}")
    for param, value in search.best_params_.items():
        print(f"  {param:20s}: {value}")
    print(f"\nBest CV F1 Score: {search.best_score_:.4f}")
    
    # Evaluate on validation set
    best_lr = search.best_estimator_
    y_val_pred = best_lr.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(f"\nValidation F1 Score: {val_f1:.4f}")
    print("\nValidation Classification Report:")
    print(classification_report(y_val, y_val_pred, target_names=['Not Important', 'Important']))
    
    # Print top 10 CV results for analysis
    print(f"\n{'='*80}")
    print("TOP 10 PARAMETER COMBINATIONS (by CV F1 Score)")
    print(f"{'='*80}")
    
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values('mean_test_score', ascending=False).head(10)
    
    for idx, row in cv_results.iterrows():
        print(f"\nRank {cv_results.index.get_loc(idx) + 1}:")
        print(f"  Mean CV F1: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")
        params_dict = row['params']
        for k, v in params_dict.items():
            print(f"    {k}: {v}")
    
    # Generate next iteration parameters
    print(f"\n{'='*80}")
    print("SUGGESTED PARAMETERS FOR NEXT ITERATION")
    print(f"{'='*80}")
    
    best_C = search.best_params_['C']
    print(f"""
# Based on current best: C={best_C}
param_grid_next_iteration = {{
    'C': [{best_C * 0.5:.4f}, {best_C * 0.75:.4f}, {best_C:.4f}, {best_C * 1.25:.4f}, {best_C * 1.5:.4f}, {best_C * 2:.4f}],
    'penalty': ['{search.best_params_.get('penalty', 'l2')}'],
    'solver': ['{search.best_params_['solver']}'],
    'max_iter': [{search.best_params_['max_iter']}],
    'class_weight': {[search.best_params_['class_weight']]},
}}
""")
    
    return best_lr, scaler, val_f1, search.best_params_

# ============================================================================
# 3. ENHANCED RIDGE REGRESSION TUNING
# ============================================================================
def train_ridge_extensive(X_train, y_train, X_val, y_val, search_type='grid'):
    """
    Extensive hyperparameter tuning for Ridge Classifier
    Ridge is L2-regularized linear model, excellent for high-dimensional data
    """
    print("\n" + "="*80)
    print("EXTENSIVE RIDGE CLASSIFIER HYPERPARAMETER TUNING")
    print("="*80)
    
    # Scale features (essential for Ridge)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    n_neg = sum(y_train == 0)
    n_pos = sum(y_train == 1)
    print(f"Class imbalance: {n_neg}:{n_pos} = {n_neg/n_pos:.2f}:1")
    
    if search_type == 'random':
        # Random search for comprehensive exploration
        param_distributions = {
            'alpha': np.logspace(-4, 4, 100),  # Regularization strength
            'fit_intercept': [True, False],
            'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'],
            'max_iter': [500, 1000, 2000, 5000],
            'tol': [1e-5, 1e-4, 1e-3, 1e-2],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}]
        }
        
        ridge = RidgeClassifier(random_state=42)
        
        search = RandomizedSearchCV(
            ridge, param_distributions,
            n_iter=250,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=2,
            random_state=42
        )
        
        print("Running randomized search (250 iterations)...")
        search.fit(X_train_scaled, y_train)
    
    else:
        # Multi-stage grid search
        print("\n--- STAGE 1: Coarse Grid Search ---")
        
        param_grid_stage1 = {
            'alpha': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0],
            'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga'],
            'max_iter': [1000, 2000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}]
        }
        
        ridge = RidgeClassifier(random_state=42)
        
        search_stage1 = GridSearchCV(
            ridge, param_grid_stage1,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1
        )
        
        search_stage1.fit(X_train_scaled, y_train)
        
        print(f"\nStage 1 Best Score: {search_stage1.best_score_:.4f}")
        print("Stage 1 Best Parameters:")
        for param, value in search_stage1.best_params_.items():
            print(f"  {param}: {value}")
        
        # Stage 2: Fine-tuning around best alpha
        print("\n--- STAGE 2: Fine-Tuning Grid Search ---")
        
        best_p = search_stage1.best_params_
        best_alpha = best_p['alpha']
        
        # Create focused alpha range around best value
        alpha_range = np.logspace(
            np.log10(best_alpha * 0.1),
            np.log10(best_alpha * 10),
            20
        )
        
        param_grid_stage2 = {
            'alpha': alpha_range,
            'solver': [best_p['solver']],
            'max_iter': [best_p['max_iter']],
            'class_weight': [
                best_p['class_weight'],
                'balanced',
                {0: 1, 1: 2},
                {0: 1, 1: 3},
                {0: 1, 1: 4}
            ],
            'tol': [1e-5, 1e-4, 1e-3]
        }
        
        search = GridSearchCV(
            ridge, param_grid_stage2,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1
        )
        
        search.fit(X_train_scaled, y_train)
    
    print(f"\n{'='*80}")
    print("RIDGE CLASSIFIER - BEST PARAMETERS")
    print(f"{'='*80}")
    for param, value in search.best_params_.items():
        print(f"  {param:20s}: {value}")
    print(f"\nBest CV F1 Score: {search.best_score_:.4f}")
    
    # Evaluate on validation set
    best_ridge = search.best_estimator_
    y_val_pred = best_ridge.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(f"\nValidation F1 Score: {val_f1:.4f}")
    print("\nValidation Classification Report:")
    print(classification_report(y_val, y_val_pred, target_names=['Not Important', 'Important']))
    
    # Analyze coefficients
    print(f"\n{'='*80}")
    print("TOP 15 FEATURES BY ABSOLUTE COEFFICIENT VALUE")
    print(f"{'='*80}")
    coefficients = np.abs(best_ridge.coef_[0])
    top_indices = np.argsort(coefficients)[-15:][::-1]
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank:2d}. Feature {idx:4d}: {coefficients[idx]:.6f}")
    
    # Print top 10 CV results
    print(f"\n{'='*80}")
    print("TOP 10 PARAMETER COMBINATIONS (by CV F1 Score)")
    print(f"{'='*80}")
    
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values('mean_test_score', ascending=False).head(10)
    
    for idx, row in cv_results.iterrows():
        print(f"\nRank {cv_results.index.get_loc(idx) + 1}:")
        print(f"  Mean CV F1: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")
        params_dict = row['params']
        for k, v in params_dict.items():
            print(f"    {k}: {v}")
    
    # Generate next iteration parameters
    print(f"\n{'='*80}")
    print("SUGGESTED PARAMETERS FOR NEXT ITERATION")
    print(f"{'='*80}")
    
    best_p = search.best_params_
    best_alpha = best_p['alpha']
    
    print(f"""
# Based on current best results
param_grid_next_iteration = {{
    'alpha': [{best_alpha * 0.5:.6f}, {best_alpha * 0.75:.6f}, {best_alpha:.6f}, {best_alpha * 1.25:.6f}, {best_alpha * 1.5:.6f}, {best_alpha * 2:.6f}],
    'solver': ['{best_p['solver']}'],
    'max_iter': [{best_p['max_iter']}],
    'class_weight': {[best_p['class_weight']]},
}}
""")
    
    return best_ridge, scaler, val_f1, search.best_params_

# ============================================================================
# 4. ENHANCED ELASTICNET (LASSO + RIDGE) TUNING
# ============================================================================
def train_elasticnet_extensive(X_train, y_train, X_val, y_val, search_type='grid'):
    """
    Extensive hyperparameter tuning for ElasticNet via Logistic Regression
    ElasticNet combines L1 (Lasso) and L2 (Ridge) regularization
    """
    print("\n" + "="*80)
    print("EXTENSIVE ELASTICNET (LASSO + RIDGE) HYPERPARAMETER TUNING")
    print("="*80)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    n_neg = sum(y_train == 0)
    n_pos = sum(y_train == 1)
    print(f"Class imbalance: {n_neg}:{n_pos} = {n_neg/n_pos:.2f}:1")
    
    if search_type == 'random':
        # Random search for comprehensive exploration
        param_distributions = {
            'C': np.logspace(-4, 3, 100),  # Inverse regularization strength
            'l1_ratio': np.linspace(0, 1, 21),  # 0=Ridge, 1=Lasso, 0.5=Equal mix
            'penalty': ['elasticnet'],
            'solver': ['saga'],  # Only saga supports elasticnet
            'max_iter': [1000, 2000, 3000, 5000],
            'tol': [1e-5, 1e-4, 1e-3],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}]
        }
        
        lr = LogisticRegression(random_state=42)
        
        search = RandomizedSearchCV(
            lr, param_distributions,
            n_iter=250,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=2,
            random_state=42
        )
        
        print("Running randomized search (250 iterations)...")
        search.fit(X_train_scaled, y_train)
    
    else:
        # Multi-stage grid search
        print("\n--- STAGE 1: Exploring L1_ratio (Lasso vs Ridge balance) ---")
        
        param_grid_stage1 = {
            'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
            'l1_ratio': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'penalty': ['elasticnet'],
            'solver': ['saga'],
            'max_iter': [2000],
            'class_weight': ['balanced', {0: 1, 1: 3}]
        }
        
        lr = LogisticRegression(random_state=42)
        
        search_stage1 = GridSearchCV(
            lr, param_grid_stage1,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1
        )
        
        search_stage1.fit(X_train_scaled, y_train)
        
        print(f"\nStage 1 Best Score: {search_stage1.best_score_:.4f}")
        print("Stage 1 Best Parameters:")
        for param, value in search_stage1.best_params_.items():
            print(f"  {param}: {value}")
        
        # Stage 2: Fine-tuning around best parameters
        print("\n--- STAGE 2: Fine-Tuning C and L1_ratio ---")
        
        best_p = search_stage1.best_params_
        best_C = best_p['C']
        best_l1 = best_p['l1_ratio']
        
        # Create focused ranges
        C_range = np.logspace(
            np.log10(best_C * 0.1),
            np.log10(best_C * 10),
            15
        )
        
        l1_range = np.linspace(
            max(0, best_l1 - 0.2),
            min(1, best_l1 + 0.2),
            11
        )
        
        param_grid_stage2 = {
            'C': C_range,
            'l1_ratio': l1_range,
            'penalty': ['elasticnet'],
            'solver': ['saga'],
            'max_iter': [best_p['max_iter'], 3000],
            'class_weight': [
                best_p['class_weight'],
                'balanced',
                {0: 1, 1: 2},
                {0: 1, 1: 3},
                {0: 1, 1: 4}
            ]
        }
        
        search = GridSearchCV(
            lr, param_grid_stage2,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1,
            verbose=1
        )
        
        search.fit(X_train_scaled, y_train)
    
    print(f"\n{'='*80}")
    print("ELASTICNET - BEST PARAMETERS")
    print(f"{'='*80}")
    for param, value in search.best_params_.items():
        print(f"  {param:20s}: {value}")
    
    best_l1_ratio = search.best_params_.get('l1_ratio', 0.5)
    print(f"\nRegularization Mix:")
    print(f"  L1 (Lasso) weight: {best_l1_ratio:.2%}")
    print(f"  L2 (Ridge) weight: {(1-best_l1_ratio):.2%}")
    print(f"\nBest CV F1 Score: {search.best_score_:.4f}")
    
    # Evaluate on validation set
    best_elastic = search.best_estimator_
    y_val_pred = best_elastic.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(f"\nValidation F1 Score: {val_f1:.4f}")
    print("\nValidation Classification Report:")
    print(classification_report(y_val, y_val_pred, target_names=['Not Important', 'Important']))
    
    # Analyze coefficients
    print(f"\n{'='*80}")
    print("COEFFICIENT ANALYSIS")
    print(f"{'='*80}")
    coefficients = best_elastic.coef_[0]
    print(f"Non-zero coefficients: {np.sum(coefficients != 0)} / {len(coefficients)}")
    print(f"Feature selection by L1: {np.sum(coefficients == 0)} features zeroed out")
    
    print(f"\nTOP 15 FEATURES BY ABSOLUTE COEFFICIENT VALUE")
    abs_coef = np.abs(coefficients)
    top_indices = np.argsort(abs_coef)[-15:][::-1]
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank:2d}. Feature {idx:4d}: {abs_coef[idx]:.6f} (raw: {coefficients[idx]:+.6f})")
    
    # Print top 10 CV results
    print(f"\n{'='*80}")
    print("TOP 10 PARAMETER COMBINATIONS (by CV F1 Score)")
    print(f"{'='*80}")
    
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values('mean_test_score', ascending=False).head(10)
    
    for idx, row in cv_results.iterrows():
        print(f"\nRank {cv_results.index.get_loc(idx) + 1}:")
        print(f"  Mean CV F1: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")
        params_dict = row['params']
        for k, v in params_dict.items():
            print(f"    {k}: {v}")
    
    # Generate next iteration parameters
    print(f"\n{'='*80}")
    print("SUGGESTED PARAMETERS FOR NEXT ITERATION")
    print(f"{'='*80}")
    
    best_p = search.best_params_
    best_C = best_p['C']
    best_l1 = best_p.get('l1_ratio', 0.5)
    
    print(f"""
# Based on current best results
param_grid_next_iteration = {{
    'C': [{best_C * 0.7:.6f}, {best_C * 0.85:.6f}, {best_C:.6f}, {best_C * 1.15:.6f}, {best_C * 1.3:.6f}],
    'l1_ratio': [{max(0, best_l1 - 0.1):.2f}, {best_l1:.2f}, {min(1, best_l1 + 0.1):.2f}],
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'max_iter': [{best_p['max_iter']}],
    'class_weight': {[best_p['class_weight']]},
}}
""")
    
    return best_elastic, scaler, val_f1, search.best_params_

# ============================================================================
# 5. ENSEMBLE WITH OPTIMAL WEIGHTS (LogReg + Ridge + ElasticNet)
# ============================================================================
def train_ensemble_optimized(X_train, y_train, X_val, y_val, search_type='grid'):
    """Train ensemble with hyperparameter tuning for all linear models"""
    
    print("\n" + "="*80)
    print("TRAINING OPTIMIZED LINEAR ENSEMBLE")
    print("Models: Logistic Regression + Ridge + ElasticNet")
    print("="*80)
    
    # Train individual models with extensive tuning
    lr_model, lr_scaler, lr_f1, lr_params = train_logistic_regression_extensive(
        X_train, y_train, X_val, y_val, search_type
    )
    
    ridge_model, ridge_scaler, ridge_f1, ridge_params = train_ridge_extensive(
        X_train, y_train, X_val, y_val, search_type
    )
    
    elastic_model, elastic_scaler, elastic_f1, elastic_params = train_elasticnet_extensive(
        X_train, y_train, X_val, y_val, search_type
    )
    
    # Get predictions from all models
    X_val_lr = lr_scaler.transform(X_val)
    X_val_ridge = ridge_scaler.transform(X_val)
    X_val_elastic = elastic_scaler.transform(X_val)
    
    # Test different ensemble weight combinations
    print("\n" + "="*80)
    print("OPTIMIZING ENSEMBLE WEIGHTS")
    print("="*80)
    
    # Get decision functions for Ridge (doesn't have predict_proba)
    lr_probs = lr_model.predict_proba(X_val_lr)
    ridge_scores = ridge_model.decision_function(X_val_ridge)
    # Convert Ridge scores to probability-like scores
    ridge_probs = np.column_stack([1 - ridge_scores, ridge_scores])
    ridge_probs = np.exp(ridge_probs) / np.exp(ridge_probs).sum(axis=1, keepdims=True)
    elastic_probs = elastic_model.predict_proba(X_val_elastic)
    
    best_ensemble_f1 = 0
    best_weights = (1/3, 1/3, 1/3)
    
    print("\nTesting weight combinations (grid search):")
    # Test combinations with 0.1 granularity
    for w_lr in [i/10 for i in range(0, 11)]:
        for w_ridge in [i/10 for i in range(0, 11 - int(w_lr*10))]:
            w_elastic = 1.0 - w_lr - w_ridge
            if w_elastic < 0:
                continue
                
            ensemble_probs = (w_lr * lr_probs + 
                            w_ridge * ridge_probs + 
                            w_elastic * elastic_probs)
            ensemble_pred = np.argmax(ensemble_probs, axis=1)
            f1 = f1_score(y_val, ensemble_pred, average='macro')
            
            if f1 > best_ensemble_f1:
                best_ensemble_f1 = f1
                best_weights = (w_lr, w_ridge, w_elastic)
                print(f"  ✓ New best: LR={w_lr:.1f}, Ridge={w_ridge:.1f}, Elastic={w_elastic:.1f} → F1={f1:.4f}")
    
    print(f"\n{'='*80}")
    print("OPTIMAL ENSEMBLE WEIGHTS")
    print(f"{'='*80}")
    print(f"Logistic Regression: {best_weights[0]:.1f}")
    print(f"Ridge Classifier:    {best_weights[1]:.1f}")
    print(f"ElasticNet:          {best_weights[2]:.1f}")
    print(f"\nBest Ensemble F1: {best_ensemble_f1:.4f}")
    
    print(f"\n{'='*80}")
    print("ENSEMBLE SUMMARY")
    print(f"{'='*80}")
    print(f"Logistic Regression F1: {lr_f1:.4f}")
    print(f"Ridge Classifier F1:    {ridge_f1:.4f}")
    print(f"ElasticNet F1:          {elastic_f1:.4f}")
    print(f"Ensemble F1:            {best_ensemble_f1:.4f}")
    print(f"\nImprovement over best single model: {best_ensemble_f1 - max(lr_f1, ridge_f1, elastic_f1):.4f}")
    
    # Final predictions with best weights
    ensemble_probs = (best_weights[0] * lr_probs + 
                     best_weights[1] * ridge_probs + 
                     best_weights[2] * elastic_probs)
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    
    print("\nEnsemble Classification Report:")
    print(classification_report(y_val, ensemble_pred, target_names=['Not Important', 'Important']))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_val, ensemble_pred)
    print(cm)
    
    # Save best parameters to file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    best_params = {
        'timestamp': timestamp,
        'logistic_regression': lr_params,
        'ridge_classifier': ridge_params,
        'elasticnet': elastic_params,
        'ensemble_weights': {
            'logistic_regression': best_weights[0],
            'ridge': best_weights[1],
            'elasticnet': best_weights[2]
        },
        'performance': {
            'lr_f1': float(lr_f1),
            'ridge_f1': float(ridge_f1),
            'elastic_f1': float(elastic_f1),
            'ensemble_f1': float(best_ensemble_f1)
        }
    }
    
    with open(f'best_params_{timestamp}.json', 'w') as f:
        json.dump(best_params, f, indent=2)
    
    print(f"\n✓ Best parameters saved to: best_params_{timestamp}.json")
    
    return lr_model, ridge_model, elastic_model, lr_scaler, ridge_scaler, elastic_scaler, best_weights, best_ensemble_f1

# ============================================================================
# 6. PREDICTION ON TEST SET
# ============================================================================
def predict_test_set(lr_model, ridge_model, elastic_model, 
                    lr_scaler, ridge_scaler, elastic_scaler, 
                    weights, X_test, test_ids):
    """Make predictions on test set using optimized linear ensemble"""
    
    print("\n" + "="*80)
    print("MAKING TEST PREDICTIONS")
    print("="*80)
    
    # Scale features for each model
    X_test_lr = lr_scaler.transform(X_test)
    X_test_ridge = ridge_scaler.transform(X_test)
    X_test_elastic = elastic_scaler.transform(X_test)
    
    # Get predictions from all models
    lr_probs = lr_model.predict_proba(X_test_lr)
    
    ridge_scores = ridge_model.decision_function(X_test_ridge)
    ridge_probs = np.column_stack([1 - ridge_scores, ridge_scores])
    ridge_probs = np.exp(ridge_probs) / np.exp(ridge_probs).sum(axis=1, keepdims=True)
    
    elastic_probs = elastic_model.predict_proba(X_test_elastic)
    
    # Weighted ensemble
    ensemble_probs = (weights[0] * lr_probs + 
                     weights[1] * ridge_probs + 
                     weights[2] * elastic_probs)
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    
    print(f"Test predictions: 0={sum(ensemble_pred==0)}, 1={sum(ensemble_pred==1)}")
    print(f"Prediction distribution: {sum(ensemble_pred==1)/len(ensemble_pred)*100:.1f}% positive class")
    
    print(f"\nIndividual model predictions:")
    lr_pred = np.argmax(lr_probs, axis=1)
    ridge_pred = ridge_model.predict(X_test_ridge)
    elastic_pred = np.argmax(elastic_probs, axis=1)
    
    print(f"  LogReg:    {sum(lr_pred==1)/len(lr_pred)*100:.1f}% positive")
    print(f"  Ridge:     {sum(ridge_pred==1)/len(ridge_pred)*100:.1f}% positive")
    print(f"  ElasticNet: {sum(elastic_pred==1)/len(elastic_pred)*100:.1f}% positive")
    
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': ensemble_pred
    })
    
    return submission

# ============================================================================
# 7. MAIN PIPELINE
# ============================================================================
def main():
    """Main training and prediction pipeline"""
    
    print("="*80)
    print("ENHANCED LINEAR MODELS PIPELINE")
    print("Models: Logistic Regression + Ridge + ElasticNet (Lasso + Ridge)")
    print("="*80)
    
    # Load data
    X_train, y_train, X_test, test_ids = load_and_preprocess_data(
        'train_part1.json',
        'test.json'
    )
    
    # Split training data
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )
    
    # Choose search type: 'grid' or 'random'
    # 'random' is faster and explores more of the space (recommended for first run)
    # 'grid' is more thorough but slower (recommended after narrowing parameters)
    search_type = 'grid'  # Change to 'random' for faster experimentation
    
    print(f"\nUsing {search_type.upper()} search strategy")
    print("\nThis will train and tune 3 linear models:")
    print("  1. Logistic Regression (baseline linear classifier)")
    print("  2. Ridge Classifier (L2 regularization)")
    print("  3. ElasticNet (L1 + L2 regularization, feature selection)")
    
    # Train ensemble with extensive tuning
    (lr_model, ridge_model, elastic_model, 
     lr_scaler, ridge_scaler, elastic_scaler, 
     best_weights, ensemble_f1) = train_ensemble_optimized(
        X_train_split, y_train_split,
        X_val_split, y_val_split,
        search_type=search_type
    )
    
    # Retrain on full training data with best parameters
    print("\n" + "="*80)
    print("RETRAINING ON FULL TRAINING DATA WITH BEST PARAMETERS")
    print("="*80)
    
    # Logistic Regression
    lr_scaler_final = StandardScaler()
    X_train_lr = lr_scaler_final.fit_transform(X_train)
    lr_final = LogisticRegression(**lr_model.get_params())
    lr_final.fit(X_train_lr, y_train)
    print("✓ Logistic Regression trained on full data")
    
    # Ridge
    ridge_scaler_final = StandardScaler()
    X_train_ridge = ridge_scaler_final.fit_transform(X_train)
    ridge_final = RidgeClassifier(**ridge_model.get_params())
    ridge_final.fit(X_train_ridge, y_train)
    print("✓ Ridge Classifier trained on full data")
    
    # ElasticNet
    elastic_scaler_final = StandardScaler()
    X_train_elastic = elastic_scaler_final.fit_transform(X_train)
    elastic_final = LogisticRegression(**elastic_model.get_params())
    elastic_final.fit(X_train_elastic, y_train)
    print("✓ ElasticNet trained on full data")
    
    # Predict on test set
    submission = predict_test_set(
        lr_final, ridge_final, elastic_final,
        lr_scaler_final, ridge_scaler_final, elastic_scaler_final,
        best_weights, X_test, test_ids
    )
    
    # Save submission
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f'submission_linear_ensemble_{timestamp}.csv'
    submission.to_csv(filename, index=False)
    
    print(f"\n✓ Submission saved: {filename}")
    
    print("\n" + "="*80)
    print("PIPELINE COMPLETED")
    print("="*80)
    print(f"Expected validation performance: F1 = {ensemble_f1:.4f}")
    print("\nKey improvements:")
    print("  ✓ Three complementary linear models")
    print("  ✓ Ridge with L2 regularization")
    print("  ✓ ElasticNet with L1+L2 (automatic feature selection)")
    print("  ✓ Extensive hyperparameter search")
    print("  ✓ Optimized ensemble weights")
    print("  ✓ Best parameters saved for iteration")
    print("\nModel characteristics:")
    print("  • Logistic Regression: Fast, interpretable baseline")
    print("  • Ridge: Better with correlated features, stable predictions")
    print("  • ElasticNet: Feature selection + regularization, sparse solutions")

if __name__ == "__main__":
    main()

ENHANCED LINEAR MODELS PIPELINE
Models: Logistic Regression + Ridge + ElasticNet (Lasso + Ridge)
Loading training data...
Training samples: 1530
Feature dimension: 1024
Class distribution: 0=1326, 1=204
Class imbalance ratio: 6.50:1

Loading test data...
Test samples: 500

Using GRID search strategy

This will train and tune 3 linear models:
  1. Logistic Regression (baseline linear classifier)
  2. Ridge Classifier (L2 regularization)
  3. ElasticNet (L1 + L2 regularization, feature selection)

TRAINING OPTIMIZED LINEAR ENSEMBLE
Models: Logistic Regression + Ridge + ElasticNet

EXTENSIVE LOGISTIC REGRESSION HYPERPARAMETER TUNING
Note: Using constrained grid search due to solver-penalty compatibility

Searching with lbfgs solver...
Fitting 5 folds for each of 240 candidates, totalling 1200 fits

Searching with saga solver...
Fitting 5 folds for each of 1800 candidates, totalling 9000 fits


KeyboardInterrupt: 